In [0]:
SELECT   
a.deployable_account_name
, fiscal_year
, REPLACE(REPLACE(a.fiscal_year_quarter, '\'', ''), ' ', '-') AS fiscal_year_quarter
, a.usage_date
, CASE WHEN a.usage_date = (SELECT MAX(usage_date) FROM main.gtm_gold.account_consumption_daily) THEN 'Y' ELSE 'N' END AS latest_snapshot

/* All */
, ROUND(SUM(a.dbu_dollars_t7d_avg), 2) AS dbu_dollars_t7d_avg
, ROUND(SUM(a.dbu_dollars_t28d_avg), 2) AS dbu_dollars_t28d_avg

/* Lakebase */
, ROUND(SUM(a.lakebase_dbu_dollars_ytd), 2) AS lakebase_dbu_dollars_ytd
, ROUND(SUM(a.lakebase_dbu_dollars_t7d_avg), 2) AS lakebase_dbu_dollars_t7d_avg
, ROUND(SUM(a.lakebase_dbu_dollars_t7d_avg_prev), 2) AS lakebase_dbu_dollars_t7d_prev
, ROUND(SUM(a.lakebase_dbu_dollars_t28d_avg), 2) AS lakebase_dbu_dollars_t28d_avg
, ROUND(SUM(a.lakebase_dbu_dollars_t28d_avg_prev), 2) AS lakebase_dbu_dollars_t28d_avg_prev
, ROUND(SUM(a.lakebase_dbu_dollars_t28d_sum), 2) AS lakebase_dbu_dollars_t28d_sum
, ROUND(SUM(a.lakebase_dbu_dollars_t28d_sum_prev), 2) AS lakebase_dbu_dollars_t28d_sum_prev
, CASE when(lakebase_dbu_dollars_t28d_sum > 5000 and lakebase_dbu_dollars_t28d_sum_prev > 5000) then true else false end as lakebase_activated
--, ROUND(TRY_DIVIDE(SUM(a.lakebase_dbu_dollars_t7d_avg), SUM(a.dbu_dollars_t7d_avg)) * 100, 4) AS lakebase_penetration_pct
--, ROUND(TRY_DIVIDE(SUM(a.lakebase_dbu_dollars_t28d_avg), SUM(a.dbu_dollars_t28d_avg)) * 100, 4) AS lakebase_penetration_t28d_pct

/* DWH */
, ROUND(SUM(a.dwh_dbu_dollars_ytd), 2) AS dwh_dbu_dollars_ytd
, ROUND(SUM(dwh_dbu_dollars_t7d_avg), 2) AS dwh_dbu_dollars_t7d_avg
, ROUND(SUM(dwh_dbu_dollars_t28d_avg), 2) AS dwh_dbu_dollars_t28d_avg
, ROUND(SUM(dwh_dbu_dollars_t7d_avg_prev), 2) AS dwh_dbu_dollars_t7d_avg_prev
, ROUND(SUM(dwh_dbu_dollars_t28d_avg_prev), 2) AS dwh_dbu_dollars_t28d_avg_prev
--, ROUND(TRY_DIVIDE(SUM(dwh_dbu_dollars_t7d_avg), SUM(dbu_dollars_t7d_avg)) * 100, 2) AS dwh_penetration_t7d_pct
--, ROUND(TRY_DIVIDE(SUM(dwh_dbu_dollars_t28d_avg), SUM(dbu_dollars_t28d_avg)) * 100, 2) AS dwh_penetration_t28d_pct

FROM main.gtm_gold.account_consumption_daily AS a
WHERE horizontal_and_vertical_hierarchy_concatenated_emails LIKE CONCAT('%', :ae_email, '%')
--AND a.deployable_account_name NOT IN ('Gruppo Hera', 'ALIA SERVIZI AMBIENTALI SPA', 'Snam Spa')
--AND a.usage_date = '2026-05-28' --TEST!!
--AND a.sales_subregion_level_3 = 'Italy Strategic Core'
AND a.dbu_dollars_t7d_avg > 0
AND a.fiscal_year >= 2026
AND (
  dayofweek(a.usage_date) = 6  -- Fridays
  OR a.usage_date = (SELECT MAX(usage_date) FROM main.gtm_gold.account_consumption_daily)  -- latest available day
)
GROUP BY ALL
ORDER BY dbu_dollars_t7d_avg DESC
